In [21]:
import os
import torch 
import torch.nn as nn
import torch.optim as optim
import torchvision 
import torchvision.transforms as transforms
import torch.utils.data as DataLoader

In [22]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = torchvision.datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader.DataLoader(test_dataset, batch_size=64, shuffle=False)

In [23]:
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super(CNN, self).__init__()

        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, stride=1, padding=1)# in_channels = 1 for grayscaled images, out_channels = number of filters, kernel_size = size of the filter, stride = step size, padding = 0 for no padding
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * 7 * 7, 128) # 32 filters of size 7x7 after pooling
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu3(x)
        x = self.fc2(x)

        return x


device = "cuda" if torch.cuda.is_available() else "cpu"
model = CNN().to(device)
print(model.parameters())

<generator object Module.parameters at 0x00000266F1532A40>


In [24]:
loss_func = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [25]:
epochs = 5
best_val_accuracy = float('-inf')
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        predictions = model(images)
        loss = loss_func(predictions, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)

            predictions = model(images)
            loss = loss_func(predictions, labels)

            val_loss += loss.item()
            _, predicted = torch.max(predictions.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    avg_val_loss = val_loss / len(test_loader)
    val_accuracy = 100 * correct / total
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        if not os.path.exists("./saved_models"):
            os.makedirs("./saved_models")
        torch.save(model.state_dict(), "./saved_models/best_model.pth")
    print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%")

Epoch [1/5], Train Loss: 0.4628, Val Loss: 0.3741, Val Accuracy: 86.39%
Epoch [2/5], Train Loss: 0.3060, Val Loss: 0.3249, Val Accuracy: 88.63%
Epoch [3/5], Train Loss: 0.2660, Val Loss: 0.2825, Val Accuracy: 89.71%
Epoch [4/5], Train Loss: 0.2371, Val Loss: 0.2767, Val Accuracy: 89.90%
Epoch [5/5], Train Loss: 0.2138, Val Loss: 0.2683, Val Accuracy: 90.34%
